# MechVerify-COF: Qwen baseline on Colab
Use **Runtime → Change runtime type → T4 GPU** if available. This notebook downloads Qwen onto the Colab cloud machine; it does not install Qwen on your laptop.
Run the cells in order. Results are saved to Google Drive **after each completed case**.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
output_dir = '/content/drive/MyDrive/MechVerify-COF/results'
os.makedirs(output_dir, exist_ok=True)
print('Results directory:', output_dir)


In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Switch to a GPU runtime before running Qwen; CPU inference may be very slow.')


In [ ]:
import subprocess
from pathlib import Path
repo_dir = Path('/content/MechVerify-COF')
if not repo_dir.exists():
    subprocess.run(['git','clone','https://github.com/bao-yutian/MechVerify-COF.git',str(repo_dir)],check=True)
else:
    subprocess.run(['git','-C',str(repo_dir),'pull','--ff-only'],check=True)
%cd /content/MechVerify-COF
%pip install -q -U transformers accelerate


## Smoke test (3 cases)
The first run downloads Qwen from Hugging Face. Saving the notebook does **not** keep its GPU runtime alive.


In [ ]:
!python -u -m mechverify.qwen_local --limit 3 --output /content/drive/MyDrive/MechVerify-COF/results/qwen_smoke_3.json


## All 15 cases
Run this only after the smoke test succeeds. The JSON in Drive is overwritten with a checkpoint after each completed case. An interrupted command currently needs to be restarted; completed outputs remain available for inspection.


In [ ]:
!python -u -m mechverify.qwen_local --output /content/drive/MyDrive/MechVerify-COF/results/qwen_all_15.json


In [ ]:
import json
from pathlib import Path
p = Path('/content/drive/MyDrive/MechVerify-COF/results/qwen_all_15.json')
if p.exists():
    data = json.loads(p.read_text(encoding='utf-8'))
    print('Status:', data.get('status','completed'))
    print('Completed cases:', len(data.get('predictions',[])))
    print(json.dumps(data.get('summary',{}),indent=2,ensure_ascii=False))
else:
    print('No 15-case result file yet. Run the cell above first.')
